# Loss-Grid Computer: RQ3 Workload Affinity (Colab)

Runs `scripts/rq3_workload_affinity.py` on Colab, with automated repo clone/update, Drive-backed asset access, and persistent run output export.

This notebook automates:
- Google Drive mount
- Repo clone/pull
- Dependency install
- Asset linking from Drive into repo
- RQ3 command execution
- Persistent result bundle export to Drive

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Expected layout:
#   DRIVE_ROOT/
#     assets/
#       cifar-10-batches-py/
#       cifar10-resnet20-0.pkl
#       cifar10-row-gru-0.pkl
#     rq3_runs/
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'
DRIVE_RESULTS_ROOT = f'{DRIVE_ROOT}/rq3_runs'

## 2. Clone repo and install dependencies

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/loss-grid-computer')

if not REPO_DIR.exists():
    !git clone https://github.com/hotz99/loss-grid-computer.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'torch',
    'torchvision',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'sympy>=1.13,<1.14',
])
print('Dependencies installed.')

## 3. Link assets from Drive (grid/checkpoints)

In [ ]:
import os
from pathlib import Path

assets_dst = Path('assets')
assets_src = Path(DRIVE_ASSETS_ROOT)

if assets_dst.exists() and not assets_dst.is_symlink():
    print(f'Assets already present in repo: {assets_dst.resolve()}')
elif assets_dst.is_symlink():
    print(f'Assets symlink already present: {assets_dst} -> {os.readlink(assets_dst)}')
elif assets_src.exists():
    assets_dst.symlink_to(assets_src)
    print(f'Linked assets from Drive: {assets_dst} -> {assets_src}')
else:
    assets_dst.mkdir(exist_ok=True)
    print(f'WARNING: {assets_src} not found. Create it in Drive and add required assets.')

required = [
    'assets/cifar-10-batches-py',
    'assets/cifar10-resnet20-0.pkl',
    'assets/cifar10-row-gru-0.pkl',
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise FileNotFoundError('Missing required assets:\n' + '\n'.join(missing))

print('Required assets are present.')

## 4. Verify runtime

In [ ]:
import os
import torch

print({'cuda_available': torch.cuda.is_available(), 'mps_available': bool(hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())})
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CPU cores: {os.cpu_count()}')
print(f'Torch: {torch.__version__}')

## 5. RQ3 command configuration

In [ ]:
DEVICE = 'auto'
INCLUDE_RESNET = True
SAMPLE_COUNT = 1024
GRID_RESOLUTION = 8
GRID_SCALE = 1.0
GPU_BATCH_SIZE = 64
CPU_BATCH_SIZE = 4
CPU_WORKERS = 2
MAX_CPU_WORKER_CANDIDATE = 2
MIN_SLOWDOWN = 1.0
MAX_SLOWDOWN = 20.0
SLOWDOWN_STEP = 1.8
OUTPUT_PATH = 'outputs/rq3/row-gru-affinity-summary.json'

print({
    'device': DEVICE,
    'include_resnet': INCLUDE_RESNET,
    'sample_count': SAMPLE_COUNT,
    'grid_resolution': GRID_RESOLUTION,
    'grid_scale': GRID_SCALE,
    'gpu_batch_size': GPU_BATCH_SIZE,
    'cpu_batch_size': CPU_BATCH_SIZE,
    'cpu_workers': CPU_WORKERS,
    'max_cpu_worker_candidate': MAX_CPU_WORKER_CANDIDATE,
    'min_slowdown': MIN_SLOWDOWN,
    'max_slowdown': MAX_SLOWDOWN,
    'slowdown_step': SLOWDOWN_STEP,
    'output': OUTPUT_PATH,
})

## 6. Run RQ3 workload-affinity script

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    'scripts/rq3_workload_affinity.py',
    '--device', DEVICE,
    '--include-resnet' if INCLUDE_RESNET else '--no-include-resnet',
    '--sample-count', str(SAMPLE_COUNT),
    '--grid-resolution', str(GRID_RESOLUTION),
    '--grid-scale', str(GRID_SCALE),
    '--gpu-batch-size', str(GPU_BATCH_SIZE),
    '--cpu-batch-size', str(CPU_BATCH_SIZE),
    '--cpu-workers', str(CPU_WORKERS),
    '--max-cpu-worker-candidate', str(MAX_CPU_WORKER_CANDIDATE),
    '--min-slowdown', str(MIN_SLOWDOWN),
    '--max-slowdown', str(MAX_SLOWDOWN),
    '--slowdown-step', str(SLOWDOWN_STEP),
    '--output', OUTPUT_PATH,
]

print('Running command:\n' + ' '.join(cmd))
subprocess.check_call(cmd)

local_output_path = Path(OUTPUT_PATH)
if not local_output_path.exists():
    raise FileNotFoundError(f'Expected output not found: {local_output_path}')

summary = json.loads(local_output_path.read_text(encoding='utf-8'))
print(f'Run status: {summary.get("status")}')
print(f'Output path: {local_output_path.resolve()}')

## 7. Persist run bundle to Drive

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = Path(DRIVE_RESULTS_ROOT) / timestamp
run_dir.mkdir(parents=True, exist_ok=True)

drive_summary_path = run_dir / 'row-gru-affinity-summary.json'
shutil.copy2(local_output_path, drive_summary_path)

manifest = {
    'created_at': timestamp,
    'repo_dir': str(REPO_DIR),
    'drive_root': DRIVE_ROOT,
    'run_dir': str(run_dir),
    'local_summary_path': str(local_output_path),
    'drive_summary_path': str(drive_summary_path),
    'config': {
        'device': DEVICE,
        'include_resnet': INCLUDE_RESNET,
        'sample_count': SAMPLE_COUNT,
        'grid_resolution': GRID_RESOLUTION,
        'grid_scale': GRID_SCALE,
        'gpu_batch_size': GPU_BATCH_SIZE,
        'cpu_batch_size': CPU_BATCH_SIZE,
        'cpu_workers': CPU_WORKERS,
        'max_cpu_worker_candidate': MAX_CPU_WORKER_CANDIDATE,
        'min_slowdown': MIN_SLOWDOWN,
        'max_slowdown': MAX_SLOWDOWN,
        'slowdown_step': SLOWDOWN_STEP,
        'output': OUTPUT_PATH,
    },
    'status': summary.get('status'),
    'platform': summary.get('platform'),
}
manifest_path = run_dir / 'run-manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print(f'Run bundle saved to {run_dir}')
print(f'Summary: {drive_summary_path}')
print(f'Manifest: {manifest_path}')

## 8. Quick metrics view

In [ ]:
rq3 = summary.get('rq3_minimal_metrics', {})
if not rq3:
    print('No rq3_minimal_metrics in summary.')
else:
    print(json.dumps(rq3, indent=2, sort_keys=True))